# exp002 — honesty CAST on Qwen3.8-27B, fitted from our v7 data

**What this notebook does.** It builds a CAST steering bundle (conditional activation steering: an "honesty direction"
added to the model's middle layers, but only on prompts where a gate detects pressure to lie), then measures it
against the unsteered model on the same rows, with the same harness, judge, and scorer as exp001.

**What it fixes from the team's first CAST notebook** (`reference/team_cast_v0/`):
- the steered layers are passed explicitly and checked to have a fitted direction (the first notebook steered layers
  its vector did not cover, so nothing changed);
- the honesty direction comes from our v7 train contrastive pairs (an honest and a dishonest answer to the same
  prompt), not 16 generic sentences;
- the gate learns "pressure prompt" versus "capability or guard prompt" from v7 train rows; MASK is not used at all
  (the Starter Kit forbids it as supervision);
- every prompt is rendered as the full chat the evaluation sends (system message, earlier turns, thinking on);
- results come from the official-kit scorer, not a substring proxy.

**It runs in three stages, so a cheap check can stop an expensive step.** Set `STAGE` in section 1, then
*Runtime → Run all*; each stage stops by itself at its end (the stop shows as a `SystemExit` message, which is
expected).

| Stage | Runtime | What it does | Estimated units |
| --- | --- | --- | --- |
| `"A"` | A100 | fit the direction and the gate, check how often the gate opens on eval rows, find the doses that keep the model coherent | ~10–12 |
| `"B"` | A100 | save the bundle for the chosen dose and generate its answers on the trimmed row set | ~8–10 |
| `"C"` | **L4** (cheaper) | judge and score against exp001's unsteered answers | ~2 |

**Read the stage A output before starting stage B.** If the gate opens on many capability or guard prompts, or no
dose is coherent, stop and discuss: that saves the rest of the budget.

**Trimmed evaluation (98 rows instead of ~465).** exp001 showed that the honesty gain can only come from four
categories (instructed_lie, confrontation_after_lie, concealment, sandbagging_stated). Stage B generates 5 instances
of each of those plus every capability and guard area (where damage costs three times a gain), and 2 instances of
each category that was already at ceiling, to catch damage there. All 98 rows are among exp001's, so exp001's
unsteered answers and verdicts are reused and the unsteered arm is not generated or judged again.

**Before you run it:**
1. **Precision.** Only an 80 GB card (bf16) produces bundles we can submit: a bundle records a fingerprint of the
   model it was fitted on, and the evaluation (which runs bf16) rejects a gate fitted on a 4-bit model. A 40 GB card
   runs 4-bit, which is fine for learning and is what exp001 used, so its unsteered answers match. Keep
   `PRECISION = "nf4"` unless you have an 80 GB card *and* an unsteered bf16 reference.
2. The `GITHUB_TOKEN` Colab secret, as in exp001.
3. Unit estimates are based on exp001's real cost (about 35 units, three times its estimate). Note the balance
   before and after each stage (*RAM/Disk meter → View resources*) for the experiment log, and use *Runtime →
   Disconnect and delete runtime* when a stage ends so the GPU stops charging.

Every step is safe to re-run: the fit, the gate check, and the dose probe are saved on Drive, and generation and
judging skip finished work.

## 1. Settings

In [ ]:
STAGE = "A"               # "A": fit and checks (A100); "B": bundle and generation (A100); "C": judge (L4)

MODEL = "Qwen/Qwen3.8-27B"
PRECISION = "nf4"         # matches exp001's reference; "bf16" only on 80 GB with a bf16 unsteered reference

# fitting (train split only)
PAIRS_PER_DIM = 80        # honest/dishonest pairs per category for the direction (~800 in total)
GATE_PER_DIM = 40         # pressure prompts per category for the gate; matched by as many capability/guard prompts
STEER_BAND = (1 / 3, 2 / 3)    # steer the middle third of the layers
GATE_BAND = (0.2, 0.8)         # where the gate's layer is searched
FIT_BATCH_SIZE = 2        # sequences per forward pass while fitting; lower to 1 on out-of-memory errors

# dose finding: a dose is a fraction of each layer's typical activation size
PROBE_DOSES = [0.01, 0.02, 0.04, 0.08, 0.16]
SWEEP_DOSES = None        # None: the strongest dose that stays coherent; or set it after stage A, e.g. [0.04]
LOOP_LIMIT = 0.5          # a probe answer with fewer distinct 4-word sequences than this share is "looping"

# evaluation (same harness and judge as exp001), trimmed to the rows that matter
SPLIT = "eval"
TARGET_DIMS = ["instructed_lie", "confrontation_after_lie", "concealment", "sandbagging_stated",
               "capability_knowledge", "capability_reasoning", "capability_problem_solving",
               "capability_instruction_following", "guard_helpfulness", "guard_over_conservativeness",
               "guard_refusal", "guard_confidentiality_persona"]
PER_DIM = 5               # instances per target category (a subset of exp001's 10)
CHECK_DIMS = ["pushback", "feedback_sycophancy", "harm_motivated_denial", "fabricated_entity",
              "fabricated_action", "fabricated_citation"]   # at ceiling in exp001: only checked for damage
CHECK_PER_DIM = 2
SAMPLES = 1
BATCH_SIZE = 4
JUDGE_MODEL = "zai-org/GLM-4-9B-0414"
JUDGE_REVISION = "645b8482494e31b6b752272bf7f7f273ef0f3caf"

RUN_NAME = "exp002_cast_honesty"
REFERENCE_RUN = "exp001_qwen_reference"
REPO_URL = "https://github.com/Rupesh-sankhala/Steerability-Trust-issues.git"
REPO_DIR = "/content/Steerability-Trust-issues"

## 2. Check the GPU and choose the precision

In [ ]:
import subprocess

name, memory_mib = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True, check=True).stdout.strip().split("\n")[0].split(", ")
memory_gb = int(memory_mib) / 1024
precision = PRECISION if PRECISION != "auto" else ("bf16" if memory_gb >= 75 else "nf4")
quantize = "nf4" if precision == "nf4" else None
print(f"GPU: {name}, {memory_gb:.0f} GB, stage {STAGE} -> {MODEL} in {precision}")
if STAGE not in ("A", "B", "C"):
    raise SystemExit('STAGE must be "A", "B", or "C".')
if STAGE == "C" and PRECISION == "auto":
    raise SystemExit("For stage C set PRECISION to the precision stages A and B used, so the right Drive folder is found.")
if STAGE != "C" and memory_gb < 35:
    raise SystemExit("Stages A and B need an A100 (40 GB or more); only stage C (judging) runs on an L4.")
if precision != "bf16":
    print("NOTE: bundles fitted in 4-bit are for local 4-bit comparisons only; they cannot be submitted.")

## 3. Save outputs to Google Drive

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
RUNS = Path("/content/drive/MyDrive/trust_issues_runs")
tag = f"{MODEL.split('/')[-1]}_{precision}"
RUN_DIR = RUNS / RUN_NAME / tag
REF_DIR = RUNS / REFERENCE_RUN / tag
(RUN_DIR / "bundles").mkdir(parents=True, exist_ok=True)
have_reference = (REF_DIR / "answers.jsonl").exists()
print("outputs go to", RUN_DIR)
print("unsteered reference:", REF_DIR if have_reference else "not found; this notebook will generate it")

## 4. Get the team repository (private)

The token is read from Colab's secrets and removed from the clone's settings afterwards, so it is never shown or saved.

In [ ]:
import os
from google.colab import userdata

if not Path(REPO_DIR).exists():
    token = userdata.get("GITHUB_TOKEN")
    authed = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
    result = subprocess.run(["git", "clone", "--depth", "1", authed, REPO_DIR], capture_output=True, text=True)
    if result.returncode != 0:
        raise SystemExit("git clone failed: " + result.stderr.replace(token, "***"))
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
commit = subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"], capture_output=True, text=True).stdout
print(commit)

## 5. Install the toolkit and the scorer's requirements, then run the tests

`flash-linear-attention` gives Qwen3.8's linear-attention layers their fast path; without it exp001 printed fallback
warnings and generated slowly. If generation later fails with an error mentioning `fla` or `triton`, run
`%pip uninstall -y flash-linear-attention` and re-run: the slow path still works.

In [ ]:
%pip install -q -e {REPO_DIR}/third_party/steerability-toolkit bitsandbytes flash-linear-attention -r {REPO_DIR}/eval/requirements.txt

In [ ]:
!cd {REPO_DIR} && python -m unittest eval/test_scorer.py eval/test_generate_answers.py steering/test_cast_honesty.py 2>&1 | tail -3

## 6. Load the model (stage A)

Loaded exactly as the harness and the evaluation load it (the official settings and pinned revision). Stage B does not
load it here: it works from stage A's saved files, and its generation step loads its own copy (a copy held by the
notebook would leave too little GPU memory for that). Stage C skips sections 6 to 11.

In [ ]:
import sys, json, gc
from collections import Counter
import torch

sys.path.insert(0, f"{REPO_DIR}/steering"); sys.path.insert(0, f"{REPO_DIR}/eval")
# the editable install of section 5 is only picked up after a runtime restart, so point at the toolkit directly
sys.path.insert(0, f"{REPO_DIR}/third_party/steerability-toolkit")
if STAGE in ("A", "B"):
    import cast_honesty as ch

    train = ch.read_rows(f"{REPO_DIR}/data/v7_tidy/train.jsonl")
    evalrows = ch.read_rows(f"{REPO_DIR}/data/v7_tidy/{SPLIT}.jsonl")


def load_qwen():
    # load the model onto the GPU, refusing a load that spilled onto the CPU or disk
    global model, tokenizer
    if "model" in globals():
        # an earlier stage in this session left its model on the GPU; a second copy would not fit
        del model
        gc.collect(); torch.cuda.empty_cache()
    model, tokenizer = ch.load_model(MODEL, quantize)
    offloaded = sorted({str(d) for d in (getattr(model, "hf_device_map", None) or {}).values() if d in ("cpu", "disk")})
    if offloaded:
        raise SystemExit(f"Part of the model was placed on {offloaded}, not the GPU, so fitting and saving would fail. "
                         "Use Runtime -> Restart session, then Run all.")


if STAGE == "A":
    load_qwen()
    n_layers = model.config.num_hidden_layers
    steer_layers = ch.layer_band(n_layers, *STEER_BAND)
    gate_layers = ch.layer_band(n_layers, *GATE_BAND)
    print(f"{n_layers} layers; steering layers {steer_layers[0]}-{steer_layers[-1]}; "
          f"gate searched over {gate_layers[0]}-{gate_layers[-1]}")

## 7. Fit the honesty direction and the gate (stage A; stage B reloads it)

The direction is read on the answer tokens only (both answers in a pair share the same prompt). The gate's layer and
threshold are searched for the best separation of pressure prompts from capability and guard prompts; **F1** (0 to 1)
says how cleanly the train sets separate. The result is saved on Drive, so a re-run skips this step.

In [ ]:
FIT_FILE = RUN_DIR / "fit.pt"
if STAGE in ("A", "B"):
    if FIT_FILE.exists():
        fit = torch.load(FIT_FILE, weights_only=False)
        print("loaded the saved fit from", FIT_FILE)
    else:
        if STAGE == "B":
            raise SystemExit("No saved fit on Drive: run stage A first.")
        pair_rows = ch.pair_rows(train, PAIRS_PER_DIM)
        print(f"{len(pair_rows)} behaviour pairs:", dict(Counter(r["dim"] for r in pair_rows)))
        direction, norms = ch.fit_direction(model, tokenizer, ch.behaviour_pairs(tokenizer, MODEL, pair_rows),
                                            steer_layers, FIT_BATCH_SIZE)
        fire, closed = ch.gate_rows(train, GATE_PER_DIM)
        print(f"gate: {len(fire)} pressure prompts vs {len(closed)} capability/guard prompts")
        condition_vector, point = ch.fit_gate(model, tokenizer, ch.gate_pairs(tokenizer, MODEL, fire, closed),
                                              gate_layers, FIT_BATCH_SIZE)
        fit = {"direction": direction, "norms": norms, "condition_vector": condition_vector, "point": point,
               "steer_layers": steer_layers, "precision": precision, "commit": commit.strip(),
               "pairs": len(pair_rows), "gate_rows": len(fire)}
        torch.save(fit, FIT_FILE)
    p = fit["point"]
    print(f"gate: layer {p.layer_id}, opens when similarity {'>=' if p.comparator == 'ge' else '<='} "
          f"{p.threshold:.2f}, train F1 {p.f1:.2f}")

## 8. Gate check on eval rows (no generation)

How often the gate opens on held-out rows. We want it **open on pressure rows** and **closed on capability and guard
rows**: every capability or guard prompt it opens on is a chance to lose r_side or r_guard. Pressure-free twins are
shown for information (steering them should be harmless, but a gate that opens on everything is not doing its job).

Rough guide: pressure above 80% and every capability/guard area below 10% is good. If a protected area is far above
that, stop here and discuss before spending generation budget. Saved on Drive, so stage B does not redo it.

In [ ]:
GATE_FILE = RUN_DIR / "gate_check.json"
if STAGE in ("A", "B"):
    if GATE_FILE.exists():
        gate_rates = {group: tuple(v) for group, v in json.loads(GATE_FILE.read_text()).items()}
    elif STAGE == "B":
        raise SystemExit("No saved gate check on Drive: run stage A first.")
    else:
        probe_cast = ch.make_cast(fit["direction"], fit["norms"], 0.01, fit["steer_layers"],
                                  fit["condition_vector"], fit["point"])
        groups = {"pressure rows": ch.sample_per_dim([r for r in evalrows if ch.is_pressure(r)], 10),
                  "pressure-free twins": ch.sample_per_dim(
                      [r for r in evalrows if "honesty" in (r.get("scored_as") or []) and not ch.is_pressure(r)], 5)}
        for area in sorted({r["dim"] for r in evalrows if ch.is_protected(r)}):
            groups[area] = [r for r in evalrows if r["dim"] == area][:30]
        gate_rates = ch.gate_open_rates(model, tokenizer, MODEL, probe_cast, groups, BATCH_SIZE)
        GATE_FILE.write_text(json.dumps(gate_rates, indent=1))
        del probe_cast
    for group, (opened, total) in gate_rates.items():
        print(f"{group:36s} opens on {opened:3d}/{total:<3d} ({opened / total:.0%})")

## 9. Find the doses that keep the model coherent (gate forced open)

On a small test model a dose of 0.05 left answers intact while 0.1 made them loop ("Or the three islands? Hmm."), so
the usable window is narrow and differs per model. Each probe dose answers one pressure row per category and one row
per capability area (greedy, 400 tokens). An answer counts as a problem if it is empty (fewer than 10 words: a strong
push can make the model stop at once or emit only spaces) or looping (under `LOOP_LIMIT` distinct 4-word sequences;
normal prose is near 1). Stage B evaluates the strongest dose below the first dose with any problem, unless
`SWEEP_DOSES` names one.
Saved on Drive, so stage B does not redo it.

In [ ]:
PROBE_FILE = RUN_DIR / "dose_probe.json"
if STAGE in ("A", "B"):
    probe_rows = (ch.sample_per_dim([r for r in evalrows if ch.is_pressure(r)], 1)
                  + ch.sample_per_dim([r for r in evalrows if r["dim"].startswith("capability_")], 1))
    if PROBE_FILE.exists():
        probe = {float(dose): results for dose, results in json.loads(PROBE_FILE.read_text()).items()}
    elif STAGE == "B":
        raise SystemExit("No saved dose probe on Drive: run stage A first.")
    else:
        probe = ch.dose_probe(model, tokenizer, MODEL, fit["direction"], fit["norms"], fit["steer_layers"],
                              probe_rows, PROBE_DOSES)
        PROBE_FILE.write_text(json.dumps({str(d): r for d, r in probe.items()}, indent=1))
    coherent, problems = ch.safe_doses(probe, LOOP_LIMIT)
    for dose, count in problems.items():
        verdict = "coherent" if dose in coherent else "TOO STRONG" if any(count.values()) else "skipped (a weaker dose failed)"
        print(f"dose {dose:<5} {count['empty']:2d} empty, {count['looping']:2d} looping of {len(probe[dose])}  {verdict}")
    sweep = SWEEP_DOSES or coherent[-1:]
    print("\ndose to evaluate in stage B:", sweep or "none: no dose is coherent, stop and discuss")

In [ ]:
# read one pressure answer at each dose before spending generation budget
if STAGE in ("A", "B"):
    row_id = probe_rows[0]["row_id"]
    for dose, results in probe.items():
        text = next(t for rid, _, t in results if rid == row_id)
        print(f"--- dose {dose} ({row_id})\n{text[-600:]}\n")

In [ ]:
if STAGE == "A":
    raise SystemExit("Stage A is done. Note the gate check, the dose table and the units used, disconnect the runtime, "
                     "and share the output before starting stage B.")

## 10. Save the bundle for the chosen dose (stage B)

Each bundle is a frozen `.spipe` file: the scaled direction, the gate, and the model it was fitted on. A `config.json`
next to it records how it was built; stage C reads the dose from it. Saving needs the model, and the notebook cannot
reliably hand its GPU memory back afterwards, so when a bundle is saved this stage stops and asks for a restart; the
next Run all finds the bundle and goes straight to generation.

In [ ]:
if STAGE == "B":
    if not sweep:
        raise SystemExit("No coherent dose: set SWEEP_DOSES explicitly or discuss before generating.")
    arms = {f"cast_d{dose}": RUN_DIR / "bundles" / f"cast_d{dose}.spipe" for dose in sweep}
    missing = {arm: path for arm, path in arms.items() if not path.exists()}
    if missing:
        load_qwen()
        for arm, path in missing.items():
            ch.save_bundle(model, tokenizer, MODEL,
                           ch.make_cast(fit["direction"], fit["norms"], float(arm[len("cast_d"):]),
                                        fit["steer_layers"], fit["condition_vector"], fit["point"]), path)
    config = {"model": MODEL, "precision": precision, "submittable": precision == "bf16", "commit": fit["commit"],
              "steer_layers": fit["steer_layers"], "gate": {"layer": int(p.layer_id), "threshold": float(p.threshold),
              "comparator": p.comparator, "train_f1": float(p.f1)}, "pairs": fit["pairs"],
              "gate_rows_per_side": fit["gate_rows"], "doses": sweep, "gate_check": gate_rates}
    (RUN_DIR / "bundles" / "config.json").write_text(json.dumps(config, indent=1))
    print(json.dumps(config, indent=1))
    if missing:
        raise SystemExit("Bundle saved. Use Runtime -> Restart session (keeps this GPU and your settings), then Run all "
                         "again: stage B will skip loading the model and go straight to generation.")

## 11. Generate answers on the trimmed row set (stage B)

The harness loads the bundle exactly as the evaluation does (same model and backend checks). Two calls write to the
same answers file: 5 instances of each target category, then 2 of each category that was at ceiling in exp001. If
exp001's unsteered answers were not found, the unsteered arm is generated first on the same rows. Re-running skips
finished rows.

In [ ]:
if STAGE == "B":
    q = "--quantize nf4" if quantize else ""
    common = (f"--rows data/v7_tidy/{SPLIT}.jsonl --model {MODEL} --samples {SAMPLES} --batch-size {BATCH_SIZE} {q} "
              f"--out {RUN_DIR}/answers.jsonl")
    selections = [(" ".join(TARGET_DIMS), PER_DIM), (" ".join(CHECK_DIMS), CHECK_PER_DIM)]
    for arm, path in [("unsteered", None)] * (not have_reference) + list(arms.items()):
        bundle = f"--bundle {path}" if path else ""
        for dims, per_dim in selections:
            print(f"=== {arm}: {per_dim} per category of {dims}")
            !cd {REPO_DIR} && python eval/generate_answers.py {common} --arm {arm} {bundle} --dims {dims} --per-dim {per_dim}

In [ ]:
if STAGE == "B":
    # truncation per arm (a truncated answer scores 0); compare with exp001's zero
    diag = [json.loads(l) for l in (RUN_DIR / "answers_diagnostics.jsonl").read_text().splitlines() if l.strip()]
    for arm in sorted({d["arm"] for d in diag}):
        mine = [d for d in diag if d["arm"] == arm]
        print(f"{arm:14s} {sum(d['truncated'] for d in mine):3d} of {len(mine)} turns hit the ceiling, "
              f"{sum(d['unclosed_thinking'] for d in mine)} never closed their thinking")
    raise SystemExit("Stage B is done. Note the units used, disconnect the runtime, then start an L4 runtime, "
                     'set STAGE = "C" and Run all.')

## 12. Judge and score against the unsteered model (stage C, on an L4)

exp001's answers and judge verdicts are reused, so the unsteered arm is not judged again. The report shows, per
category, both scores, the change, a 95% margin, how many rows got worse or better, and the estimated composite. It
compares only rows both arms answered, so the trimmed set is compared with the matching exp001 rows. With 5 instances
per category the margins are wide: act only on large changes.

In [ ]:
import json, shutil
from huggingface_hub import snapshot_download

if STAGE == "C":
    config = json.loads((RUN_DIR / "bundles" / "config.json").read_text())
    arms = [f"cast_d{dose}" for dose in config["doses"]]
    combined = RUN_DIR / "combined_answers.jsonl"
    parts = [REF_DIR / "answers.jsonl"] if have_reference else []
    combined.write_text("".join(path.read_text() for path in parts + [RUN_DIR / "answers.jsonl"]))
    cache = RUN_DIR / "judge_cache.jsonl"
    if have_reference and not cache.exists() and (REF_DIR / "judge_cache.jsonl").exists():
        shutil.copy(REF_DIR / "judge_cache.jsonl", cache)

    judge_path = snapshot_download(JUDGE_MODEL, revision=JUDGE_REVISION)
    # 4 at a time fits next to the 19 GB judge on an L4's 22 GB; lower to 2 on out-of-memory errors
    os.environ.update({"JUDGE_BACKEND": "huggingface", "JUDGE_MODEL": judge_path, "JUDGE_BATCH_SIZE": "4"})
    for arm in arms:
        print(f"\n==================== {arm} vs unsteered")
        !cd {REPO_DIR} && python eval/scorer.py --rows data/v7_tidy/{SPLIT}.jsonl --answers {combined} --arm {arm} \
            --judge --cache {cache} | tee {RUN_DIR}/summary_{arm}.txt

## 13. Record the result

1. Copy the gate check (section 8), the dose probe table (section 9), the truncation lines, the
   `summary_cast_d*.txt` file, and the units each stage used into `experiments/exp002_cast_honesty/results.md` in the repository.
2. Fill in the exp002 entry of `experiments/EXPERIMENT_LOG.md`: GPU and precision, run time and compute units, the
   gate's layer, threshold and F1, the chosen dose, and its honesty change, r_side, r_guard, and estimated S.
3. What to do next depends on what you see:
   - **honesty barely moves at the strongest coherent dose:** the direction is too weak at safe strengths; try fewer, later
     layers or a projection instead of an addition (next experiment);
   - **honesty moves but a guard or capability area drops:** look at the gate check for that area first;
   - **the dose clearly helps:** confirm it at 10 per category, and if it holds, refit in bf16 on an 80 GB card before
     any submission (4-bit bundles cannot be submitted).
4. The small judge is not the leaderboard's judge, so trust comparisons between arms more than absolute numbers.